# Pattern 2: Parallelization

**Idea:** when sub-tasks don't depend on each other, run the model calls **at the same time** and combine the results in code. There are two flavors:

- **Sectioning:** split the work into *different* independent pieces (security review, performance review, readability review) and run them in parallel.
- **Voting:** run the *same* prompt several times and aggregate the answers (majority vote) to get a more reliable decision.

```
              ┌─▶ [LLM: security]    ─┐
code snippet ─┼─▶ [LLM: performance] ─┼─▶ merge in Python ─▶ report
              └─▶ [LLM: readability] ─┘
```

**Why it exists:** (1) speed: three 5-second calls in parallel take about 5 seconds, not 15. (2) focus: a reviewer that only looks at security does a better security review than one juggling three concerns. (3) confidence: voting smooths out one-off mistakes.

**Trade-off:** you pay for every call, and you need a merge step. Voting multiplies cost by N.

**Java analogy:** `CompletableFuture.supplyAsync(...)` for each reviewer, then `CompletableFuture.allOf(...).join()`. In Python we use `ThreadPoolExecutor`, which is almost the same thing as Java's `ExecutorService`.

## Step 0: Setup (same in every notebook)

Every notebook starts with the same three setup cells, so you can open any of them alone without running the others first.

1. **Load configuration** from the `.env` file in the repo root (three variables: `PORTKEY_BASE_URL`, `PORTKEY_API_KEY`, `MODEL`).
2. **Create the client.** We use the official `anthropic` Python SDK and point it at the Portkey gateway instead of `api.anthropic.com`.
3. **Define a helper**, `call_claude()`, that wraps one API call and prints everything: the prompt going in, the text coming out, token usage, and timing.

> **Java analogy:** `.env` + `python-dotenv` is like `application.properties` + Spring's `@Value`. The client is a `RestTemplate`/`WebClient` bean configured with a base URL and a default header. Portkey sits in between like an API gateway (think Spring Cloud Gateway): it holds the real Anthropic credentials, routes the call, and logs it.

In [ ]:
# --- Cell 0.1: load configuration from .env ---------------------------------
import os
import json
import time
from dotenv import load_dotenv, find_dotenv

# find_dotenv(usecwd=True) walks UP from the notebook's folder until it finds a .env file.
# That's why the .env can live in the repo root while notebooks live in notebooks/.
env_path = find_dotenv(usecwd=True)
print(f"[setup] .env file found at: {env_path or 'NOT FOUND (using existing environment variables)'}")
load_dotenv(env_path, override=False)  # override=False: real env vars win over the file

PORTKEY_BASE_URL = os.getenv("PORTKEY_BASE_URL", "https://api.portkey.ai")
PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY")
MODEL = os.getenv("MODEL")

# Fail fast with a clear message instead of a confusing HTTP 401 later.
assert PORTKEY_API_KEY, "PORTKEY_API_KEY is missing. Copy .env.example to .env and fill it in."
assert MODEL, "MODEL is missing. Example: @anthropic/claude-sonnet-4-5-20250929"

# Never print a full secret. Show just enough to confirm which key is loaded.
masked_key = PORTKEY_API_KEY[:4] + "..." + PORTKEY_API_KEY[-4:]
print(f"[setup] PORTKEY_BASE_URL = {PORTKEY_BASE_URL}")
print(f"[setup] PORTKEY_API_KEY  = {masked_key}")
print(f"[setup] MODEL            = {MODEL}")

In [ ]:
# --- Cell 0.2: create the Anthropic client, routed through Portkey -----------
import anthropic

client = anthropic.Anthropic(
    # The SDK requires an api_key, but Portkey holds the real Anthropic key for us.
    # Portkey identifies us through the x-portkey-api-key header instead.
    api_key="not-used-portkey-handles-auth",
    base_url=PORTKEY_BASE_URL,
    default_headers={"x-portkey-api-key": PORTKEY_API_KEY},
)
print(f"[setup] Anthropic SDK version: {anthropic.__version__}")
print(f"[setup] Client ready. Every request goes to {PORTKEY_BASE_URL}/v1/messages")

In [ ]:
# --- Cell 0.3: a logging helper around ONE model call -------------------------
# Everything in these notebooks is built from this single building block.
# The patterns are not new API features: they are just different ways of
# arranging plain calls like this one with ordinary Python code.

def call_claude(prompt, system=None, label="call", max_tokens=1024, verbose=True):
    """Send one user message to Claude and return the response text.

    prompt     -> the user message (str)
    system     -> optional system prompt: the model's role and rules
    label      -> a name for this call so the logs are easy to follow
    verbose    -> print the full prompt and response (set False inside threads)
    """
    if verbose:
        print("=" * 78)
        print(f"[{label}] >>> REQUEST")
        if system:
            print(f"[{label}] system: {system}")
        print(f"[{label}] user:\n{prompt}")

    started = time.perf_counter()
    kwargs = {"model": MODEL, "max_tokens": max_tokens,
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    elapsed = time.perf_counter() - started

    # response.content is a LIST of blocks. For plain text calls there is one text block.
    text = "".join(block.text for block in response.content if block.type == "text")

    if verbose:
        print(f"[{label}] <<< RESPONSE ({elapsed:.2f}s, "
              f"in={response.usage.input_tokens} tok, out={response.usage.output_tokens} tok, "
              f"stop_reason={response.stop_reason})")
        print(text)
        print("=" * 78)
    if response.stop_reason == "max_tokens":
        print(f"[{label}] WARNING: response was cut off by max_tokens")
    return text


def parse_json(text):
    """Models sometimes wrap JSON in ```json fences. Strip them, then parse."""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(cleaned)

print("[setup] call_claude() and parse_json() are defined")

In [ ]:
# --- Cell 0.4: smoke test. If this works, your .env and gateway are fine -------
call_claude("Reply with exactly: gateway OK", label="smoke-test", max_tokens=20)

## Step 1: The input, a small piece of code to review

In [ ]:
snippet = """
def get_user(user_id):
    query = "SELECT * FROM users WHERE id = " + user_id
    rows = db.execute(query)
    result = []
    for r in rows:
        for r2 in rows:
            if r == r2:
                result.append(r)
    return result
"""
print(snippet)

## Step 2: Sectioning, one focused reviewer per concern

Each reviewer gets a **different system prompt**. They don't know about each other.

We set `verbose=False` inside threads because three threads printing at once would interleave into an unreadable mess. Instead each worker logs a short start/finish line with a timestamp so you can *see* they overlap.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import threading

aspects = {
    "security":    "You are a security reviewer. Only report security problems. Max 3 bullet points.",
    "performance": "You are a performance reviewer. Only report performance problems. Max 3 bullet points.",
    "readability": "You are a readability reviewer. Only report naming/clarity problems. Max 3 bullet points.",
}

t0 = time.perf_counter()

def review(aspect):
    """Runs in a worker thread. Same as a Java Callable<String>."""
    print(f"[{time.perf_counter()-t0:5.2f}s] START  {aspect:12} on {threading.current_thread().name}")
    text = call_claude(snippet, system=aspects[aspect], label=aspect, verbose=False)
    print(f"[{time.perf_counter()-t0:5.2f}s] FINISH {aspect:12}")
    return aspect, text

# max_workers=3: one thread per reviewer. executor.map keeps the input order.
with ThreadPoolExecutor(max_workers=3) as executor:
    results = dict(executor.map(review, aspects))

print(f"\n[parallel] all {len(results)} reviews done in {time.perf_counter()-t0:.2f}s total")

## Step 3: Merge (plain Python)

The merge here is simple string assembly. It could also be another model call ("summarize these three reviews"), which would make it a tiny chain after the parallel step.

In [ ]:
report = []
for aspect, text in results.items():
    report.append(f"## {aspect.upper()}\n{text}\n")
print("\n".join(report))

## Step 4: Compare with sequential timing

Same three calls, one after another. The difference in wall-clock time is the whole point of this pattern.

In [ ]:
t_seq = time.perf_counter()
for aspect in aspects:
    call_claude(snippet, system=aspects[aspect], label=aspect, verbose=False)
    print(f"[sequential] {aspect:12} done at {time.perf_counter()-t_seq:5.2f}s")
print(f"[sequential] total {time.perf_counter()-t_seq:.2f}s (compare with the parallel total above)")

## Step 5: Voting, same question asked N times

Now a *decision* task: "Is this code safe to merge?" One answer might be a fluke. Five answers with a majority vote are more robust.

The votes come out different because the model samples tokens with some randomness. Voting turns that randomness into a confidence signal: 5/5 YES is a stronger result than 3/5 YES.

In [ ]:
from collections import Counter

vote_system = "You are a strict code reviewer. Answer with exactly one word: YES (safe to merge) or NO."
vote_prompt = f"Is this code safe to merge to production?\n{snippet}"
N_VOTES = 5

def vote(i):
    answer = call_claude(vote_prompt, system=vote_system, label=f"vote-{i}", max_tokens=5, verbose=False)
    answer = answer.strip().upper().rstrip(".")
    print(f"[vote-{i}] {answer}")
    return answer

with ThreadPoolExecutor(max_workers=N_VOTES) as executor:
    votes = list(executor.map(vote, range(1, N_VOTES + 1)))

tally = Counter(votes)
winner, count = tally.most_common(1)[0]
print(f"\n[voting] tally = {dict(tally)}")
print(f"[voting] decision = {winner} ({count}/{N_VOTES} agree)")

## Recap

| Flavor | Calls are... | Combine with... | Good for |
|---|---|---|---|
| Sectioning | different prompts | concatenation or a summary call | independent sub-tasks |
| Voting | the same prompt N times | majority / threshold | decisions where you want confidence |

Like chaining, the **structure is fixed in your code**. The model doesn't decide how many reviewers exist. When the sub-tasks themselves must be decided at runtime, you need the orchestrator pattern (notebook 03).

## Experiments to try

1. Add a fourth aspect (`"testing"`) to `aspects` and confirm the total time barely changes.
2. Change `max_workers=3` to `max_workers=1`. It becomes sequential; check the timestamps.
3. Fix the SQL injection in `snippet` (use a parameterized query) and re-run the vote. Does the tally shift?